In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import json

dataset_path = "../dataset/train"

# ✅ IMPORTANT: Use EfficientNet preprocessing (NOT rescale=1./255)
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

print("Classes:", train_data.class_indices)

# Save class mapping
with open("../backend/classes.json", "w") as f:
    json.dump(train_data.class_indices, f)

# Load EfficientNet
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False  # Freeze first

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
predictions = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Phase 1 Training...")

model.fit(
    train_data,
    validation_data=val_data,
    epochs=8,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)

# ✅ Fine tune last 30 layers
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Phase 2 Fine-tuning...")

model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)

model.save("../backend/plant_disease_model.keras")

print("FINAL MODEL SAVED SUCCESSFULLY")

Found 6533 images belonging to 19 classes.
Found 1628 images belonging to 19 classes.
Classes: {'chili_cercospora': 0, 'chili_healthy': 1, 'chili_mites_and_trips': 2, 'chili_powdery_mildew': 3, 'mango_anthracnose': 4, 'mango_bacterial_canker': 5, 'mango_cutting_weevil': 6, 'mango_die_back': 7, 'mango_gall_midge': 8, 'mango_healthy': 9, 'mango_powdery_mildew': 10, 'mango_sooty_mould': 11, 'potato_bacteria': 12, 'potato_fungi': 13, 'potato_healthy': 14, 'potato_nematode': 15, 'potato_pest': 16, 'potato_phytophthora': 17, 'potato_virus': 18}
Phase 1 Training...
Epoch 1/8
205/205 ━━━━━━━━━━━━━━━━━━━━ 731s 3s/step - accuracy: 0.4434 - loss: 1.8982 - val_accuracy: 0.6511 - val_loss: 1.2795
Epoch 2/8
205/205 ━━━━━━━━━━━━━━━━━━━━ 593s 3s/step - accuracy: 0.6992 - loss: 1.0463 - val_accuracy: 0.7033 - val_loss: 0.9853
Epoch 3/8
205/205 ━━━━━━━━━━━━━━━━━━━━ 640s 3s/step - accuracy: 0.7474 - loss: 0.8166 - val_accuracy: 0.7224 - val_loss: 0.8688
Epoch 4/8
205/205 ━━━━━━━━━━━━━━━━━━━━ 612s 3s/step

In [2]:
import os
print(os.getcwd())

C:\Users\radhe\Desktop\Plant-Disease-Detection\model_training


In [3]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'train.ipynb']


In [6]:
from tensorflow.keras.models import load_model

# Load existing keras model
model = load_model("../backend/plant_disease_model.keras", compile=False)

# Save as H5 format
model.save("../backend/plant_disease_model.h5")

print("Converted to H5 Successfully")

Converted to H5 Successfully


In [2]:
from flask import Flask
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.route("/")
def home():
    return "Backend running"

if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\Users\radhe\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
